# US Firm Characteristics: Holdout Predictions

**Chapter 20 - Out-of-sample evaluation**

Every number in this case study so far was measured on the validation folds, and every
choice was made by looking at them: which model family, which configuration, how many
names to hold, how to size them. A result selected that way cannot also be evidence that
the selection was sound - the ranking and the evidence would be the same measurement.

The holdout is the window nothing has been selected on. This notebook fits the selected
configuration on the history available before that window opens and writes its
predictions over it. [`16_holdout_backtest`](16_holdout_backtest.ipynb) turns those
predictions into a return series with the sizing the case study settled on, and
[`17_strategy_analysis`](17_strategy_analysis.ipynb) reads both back.

**What this notebook is careful about**

A holdout prediction is not the validation model scored on a later window. That is the
mistake this case study had already made: the registry carried a holdout prediction set
generated from the same training identity as the validation run, so what it scored was a
model whose parameters had been chosen while looking at the folds it was being judged
against. Section 2 fits again, and the new training identity is what makes the refit
visible rather than asserted.

**Prerequisites:** [`14_costs`](14_costs.ipynb), which fixes the configuration the
holdout runs.

**Scope:** one training run and one prediction set. No backtest, no selection, no
comparison - those are 16 and 17.

<!-- ML4T_ZH_BEGIN c0000-p000 -->
# 美國公司特徵：保留測試集預測

**第 20 章－樣本外評估**

迄今本案例研究的每個數值都在驗證折上衡量，每項選擇也都透過觀察這些結果做出：模型家族、組態、持股檔數，以及部位大小。以這種方式選出的結果，不能同時作為該選擇合理的證據；排名與證據會是同一次測量。

保留測試集是一段未用來做任何選擇的時間視窗。本 Notebook 使用視窗開啟前可取得的歷史資料，擬合選定組態，並寫入該視窗期間的預測。[`16_holdout_backtest`](16_holdout_backtest.ipynb) 會使用本案例研究最後選定的部位配置方式，將這些預測轉成報酬序列；[`17_strategy_analysis`](17_strategy_analysis.ipynb) 則會讀回兩者。

**本 Notebook 特別留意的事項**

保留測試集預測，不是把驗證期模型拿到後續視窗評分。本案例研究先前正犯過這個錯誤：登錄資料庫中有一組保留測試集預測，使用了與驗證期執行結果相同的訓練識別，因此所評分的模型，其參數是在觀察用來評判它的那些驗證折時選定的。第 2 節會重新擬合；新的訓練識別讓重新擬合成為可觀察的事實，而不只是宣稱。

**先備條件：**[`14_costs`](14_costs.ipynb)，它確定保留測試所執行的組態。

**範圍：**一次訓練執行與一組預測。不做回測、篩選或比較；這些工作留給第 16 與第 17 份 Notebook。
<!-- ML4T_ZH_END c0000-p000 -->


In [1]:
"""US Firm Characteristics: Holdout Predictions."""

import polars as pl

from case_studies.research import open_study
from case_studies.research.holdout import build_holdout_training_spec
from case_studies.research.models import reconstruct_locked_model_request
from case_studies.utils.registry import training_hash_from_spec
from case_studies.utils.registry.maintenance import delete_prediction_generation
from case_studies.utils.strategy_analysis import (
    holdout_generations_to_retire,
    registered_holdout_generations,
    resolve_solvent_carrier,
)
from case_studies.utils.warning_policy import apply_notebook_warning_policy
from utils.paths import get_case_study_dir

apply_notebook_warning_policy()

In [2]:
CASE_STUDY_ID = "us_firm_characteristics"
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
# Whether a holdout generation for a DIFFERENT configuration may be superseded by this run.
# Off by default: see section 3.
#
# The flag exists because the holdout is not a one-shot resource, which is a ruling and not
# an oversight. What the rule against consulting the holdout forbids is SELECTING on it: the
# configuration evaluated here is chosen by validation backtest Sharpe, and no holdout number
# feeds back into that choice. It says nothing about how many times the evaluation may be
# computed, and a wrong result is deleted and re-run rather than left standing because it was
# observed. Reading the rule as a physical constraint is what produced a lock layer around
# this window, and it is being removed. The guard here is against something narrower and real:
# two generations readable at once, so nobody downstream has to choose between them and nobody
# can quote whichever number they prefer.
REPLACE_HOLDOUT = False

In [3]:
study = open_study(CASE_STUDY_ID, execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)


def _delete_holdout_generation(case_dir, prediction_hash):
    """Remove one holdout prediction set and everything registered against it.

    The rows go rather than being marked, because a holdout evaluation that is still
    readable is still a number someone can quote, and the point of removing it is that it
    should not be one.

    The cascade lives in `case_studies/utils/registry/maintenance.py` and derives the child
    tables from the schema. The version this replaces listed them by hand and was already
    missing `cohort_metrics.leader_hash`, which with foreign keys enabled aborts the delete
    rather than orphaning a row - so on any registry with a cohort this function raised
    instead of deleting, and the generation it was called on stayed.
    """
    deleted = delete_prediction_generation(case_dir / "run_log" / "registry.db", prediction_hash)
    for table, n in sorted(deleted.items()):
        print(f"  deleted {n:>3} from {table}")

## 1. Which configuration the holdout runs

The holdout runs the configuration the case study reports, resolved through the same
`resolve_solvent_carrier` [`14_costs`](14_costs.ipynb) uses. Resolving it again here
rather than passing it along is deliberate: the two notebooks must agree by construction,
and a hash written down in one and read in the other agrees only until the sweep is
rebuilt.

Nothing about the holdout enters this choice. The selected configuration is the validation
rank-1, and it was fixed before this notebook ran.

<!-- ML4T_ZH_BEGIN c0004-p000 -->
## 1. 保留測試所執行的組態

保留測試會執行本案例研究要報告的組態，並透過 `resolve_solvent_carrier` 解析；這與 [`14_costs`](14_costs.ipynb) 所用的方式相同。此處刻意重新解析，而非從前一份 Notebook 傳遞：兩份 Notebook 必須依設計取得相同結果；若在一處記下雜湊值、另一處讀取，只有在掃描尚未重建前才會一致。

此項選擇完全不使用保留測試集。選中組態是驗證期排名第一的結果，在本 Notebook 執行前就已確定。
<!-- ML4T_ZH_END c0004-p000 -->


In [4]:
carrier = resolve_solvent_carrier(CASE_STUDY_ID)
print(
    f"Selected configuration: {carrier['val_backtest_hash']}  stage={carrier['val_stage']}  "
    f"family={carrier['family']}  config={carrier['config_name']}  "
    f"label={carrier['label']}"
)
print(
    f"  validation Sharpe {carrier['val_sharpe']:.3f}, max drawdown {carrier['max_drawdown']:.3f}"
)
print(f"  fitted by training run {carrier['training_hash']}")

Selected configuration: f8a96bc86f1c  stage=allocation  family=gbm  config=leaves_31_huber  label=fwd_ret_1m_win
  validation Sharpe 3.682, max drawdown -0.177
  fitted by training run 7bc69205868d


The checkpoint is part of the configuration. This family publishes a prediction set per boosting
iteration on a declared schedule, and the selected configuration's prediction set names one of
them - so refitting without it would produce a model at the end of training rather than the one
that was ranked.

<!-- ML4T_ZH_BEGIN c0006-p000 -->
檢查點是組態的一部分。此模型家族會依宣告的排程，在每次梯度提升迭代時發布一組預測；選中組態的預測集指定了其中一個檢查點。因此若重新擬合時未帶入該檢查點，就會得到訓練結束時的模型，而非當初參與排名的模型。
<!-- ML4T_ZH_END c0006-p000 -->


In [5]:
validation_prediction = study.results.open(carrier["val_prediction_hash"])
prediction_record = validation_prediction.registry_record()
CHECKPOINT_KIND = prediction_record["checkpoint_kind"]
CHECKPOINT_VALUE = prediction_record["checkpoint_value"]
print(f"Checkpoint: {CHECKPOINT_KIND}={CHECKPOINT_VALUE}")

Checkpoint: iteration=500


## 2. The window, and the model that is allowed to see it

The holdout window is not a choice made here. It is `evaluation.holdout_start` and
`evaluation.holdout_end` from the case study's own `setup.yaml`, read through the same
`canonical_window` the fold derivation and the backtest slice both go through, so the
three cannot disagree.

The training interval is everything available before that window, bounded above by a
label buffer. The buffer is what stops the last training label's outcome from resolving
inside the holdout: this case study dates each row by the month the return was earned,
so a monthly label observed at the end of December is already realised, and the buffer
is one observation rather than a horizon's worth. A zero gap would be a leak, not a
conservative choice, so the derivation refuses to default it.

Everything else about the configuration is carried across unchanged, and the fields that
cannot be - the eligibility manifest, and any parameter this family resolves from a
fold's own training rows - are recomputed against the holdout fold. Carrying those
forward would fit a model keyed to the validation folds and call it a retrain.

<!-- ML4T_ZH_BEGIN c0008-p000 -->
## 2. 時間視窗，以及允許查看它的模型

保留測試視窗不是在此做出的選擇。它是 `evaluation.holdout_start` 與 `evaluation.holdout_end`，來自案例研究自身的 `setup.yaml`，並透過折數推導與回測切片共同使用的 `canonical_window` 讀取，因此三者不會出現分歧。

訓練區間涵蓋該視窗前所有可用資料，其上界受到標籤緩衝區限制。緩衝區可防止最後一筆訓練標籤的結果落在保留測試期間才確定：本案例研究以賺得報酬的月份作為每列日期，因此在十二月底觀測到的月度標籤已實現，緩衝區只需一筆觀測值，而非整個預測期間的長度。零間隔會造成資料洩漏，並非保守選擇，因此推導程序拒絕以零作為預設值。

組態其他部分原樣沿用；不能直接沿用的欄位，包括合格資格清單，以及此模型家族依各折自身訓練資料列解析的參數，則會針對保留測試折重新計算。若直接沿用，得到的仍是綁定驗證折的模型，卻被稱作重新訓練。
<!-- ML4T_ZH_END c0008-p000 -->


In [6]:
observation_timeline = (
    pl.read_parquet(study.root / "labels" / f"{carrier['label']}.parquet")
    .get_column("timestamp")
    .unique()
    .sort()
    .to_list()
)
validation_spec = study.results.open(carrier["training_hash"]).spec()
holdout_spec = build_holdout_training_spec(
    study,
    validation_spec,
    timeline=observation_timeline,
    case_study=CASE_STUDY_ID,
)

fold = holdout_spec["computation"]["cv"]["folds"][0]
print(f"Holdout fold {fold['fold']}")
print(f"  trains  {fold['train_start']} -> {fold['train_end']}")
print(f"  predicts {fold['val_start']} -> {fold['val_end']}")
print(f"  label buffer: {holdout_spec['computation']['cv']['request']['label_buffer']}")

# The validation folds are what the buffer is measured against, and the last of them ends
# before the holdout opens. Printing both is what lets a reader check the gap rather than
# take it on the derivation's word.
validation_folds = validation_spec["computation"]["cv"]["folds"]
latest_validation_end = max(str(entry["val_end"]) for entry in validation_folds)
print(f"Validation folds: {len(validation_folds)}, latest evaluation end {latest_validation_end}")
print(f"Holdout training ends {fold['train_end']}, holdout opens {fold['val_start']}")

Holdout fold 10
  trains  1996-12-31 -> 2015-11-30
  predicts 2016-01-01 -> 2016-12-31
  label buffer: 1M
Validation folds: 10, latest evaluation end 2015-12-31 00:00:00
Holdout training ends 2015-11-30, holdout opens 2016-01-01


## 3. Fit, and register the predictions

`reconstruct_locked_model_request` builds the request from the spec above. Its name
comes from the locked holdout path this case study no longer uses; it takes a training
specification and a checkpoint, not a lock, and it is used here because it is the one
call that refuses a request that is not exactly the spec it was handed - the training
identity, the checkpoint schedule, the feature lineage and the runtime parameters are
all checked before anything is fitted.

The training identity below is new. It has to be: it covers the CV interval, and the
holdout fold is not one of the validation folds. A run that came back with the
validation training hash would mean the refit did not happen.

**The window carries one configuration at a time.** The holdout is re-runnable, and that
is not the same as free: every configuration evaluated on it is another look at a period
the case study reports as unseen, and two evaluated quietly would make that report false.

So the check below is on the selected configuration rather than on the notebook, and it has
exactly two outcomes. With the selected configuration unchanged this is an idempotent replay: the
derivation is deterministic and the training identity covers it, so the same identity comes back
and the fit is served from the registry. With the selected configuration changed it refuses,
names both configurations, and stops.


<!-- ML4T_ZH_BEGIN c0010-p000 -->
## 3. 擬合並登錄預測

`reconstruct_locked_model_request` 根據上方規格建立請求。其名稱來自本案例研究已不再使用的鎖定保留測試路徑；它接收訓練規格與檢查點，而非鎖定標記。此處使用它，是因為只有這項呼叫會拒絕與傳入規格不完全相同的請求：在開始擬合之前，會檢查訓練識別、檢查點排程、特徵來源脈絡與執行參數。

下方的訓練識別是新的，而且必須如此：它涵蓋交叉驗證區間，而保留測試折不屬於任何驗證折。若回傳的執行結果帶有驗證期訓練雜湊值，就代表重新擬合並未發生。

**此視窗一次只容納一個組態。**保留測試可以重新執行，但這不代表可以毫無代價地反覆試驗：每個在此評估的組態，都等於又看了一次被案例研究稱為未見過的期間；若悄悄評估兩個組態，這項宣稱就不成立。

因此，下方檢查針對選中組態，而非 Notebook 本身，且只有兩種結果。若選中組態未變，這是可重複執行而結果不變的重播：推導具決定性，訓練識別涵蓋該推導，因此會回傳相同識別，並從登錄資料庫提供擬合結果。若選中組態已變，檢查會拒絕執行、列出兩個組態並停止。
<!-- ML4T_ZH_END c0010-p000 -->

`REPLACE_HOLDOUT` is the only way past that, and it is a replacement rather than an
addition: the superseded generation's rows are deleted, so the registry never holds two
refits of the holdout window and no downstream resolver has to choose between them.
Deleting is what makes the earlier evaluation cost something to discard. It is also the
only honest shape - a run that had been observed and then quietly kept alongside its
replacement would let a reader take whichever number they preferred.

<!-- ML4T_ZH_BEGIN c0010-p001 -->
只有 `REPLACE_HOLDOUT` 能通過該限制，而且它進行的是取代，而非新增：被取代版本的資料列會刪除，因此登錄資料庫不會同時持有同一保留測試視窗的兩次重新擬合，下游解析器也不必在兩者之間選擇。刪除使得放棄先前評估確實有代價。這也是唯一誠實的呈現方式：若已觀察過的執行結果被悄悄保留在取代結果旁邊，讀者就能任選偏好的數字。
<!-- ML4T_ZH_END c0010-p001 -->


In [7]:
holdout_training_hash = training_hash_from_spec(holdout_spec)
this_generation = (holdout_training_hash, (CHECKPOINT_KIND, CHECKPOINT_VALUE))
retire = holdout_generations_to_retire(CASE_DIR, this_generation=this_generation)
# A row whose training run records no CV split cannot be shown either way, and deleting on
# that would discard a result nothing has established is wrong. It stops the run instead.
if retire.unattributable:
    raise RuntimeError(
        "the holdout window carries prediction sets whose training runs record no CV split, "
        "so whether they were refitted for the holdout cannot be established: "
        + ", ".join(
            f"{row['prediction_hash']} (training {row['training_hash']})"
            for row in retire.unattributable
        )
        + ". Establish what produced them before registering another evaluation on the same "
        "window; this notebook will not delete a row it cannot show is not a holdout result."
    )
# A row whose training run declares a non-holdout CV may not be reported as a holdout
# result, and it is also not something to delete unattended: `generate_holdout` refits on a
# holdout fold and then registers the predictions under the VALIDATION training identity, so
# this record covers both a validation-fitted model published over the window and a real
# refit filed under the wrong identity. Nothing owned this before - the filter here was
# `row["refitted"]`, which made exactly these rows invisible to the refusal and to
# everything after it.
if retire.not_out_of_sample and not REPLACE_HOLDOUT:
    raise RuntimeError(
        "the holdout window carries prediction sets whose training runs declare a CV split "
        "other than the holdout: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in retire.not_out_of_sample
        )
        + ". Each is either a validation-fitted model published over the window, which is "
        "not an out-of-sample result, or a refit registered under its validation training "
        "identity, which `20_strategy_synthesis/holdout.py::generate_holdout` produces - and "
        "the registry cannot tell those apart. Establish which, then set REPLACE_HOLDOUT="
        "True to remove it, or leave it and resolve the identity instead."
    )
for row in retire.not_out_of_sample:
    print(
        f"REMOVING {row['prediction_hash']} ({row['config_name']}, training "
        f"{row['training_hash']}): its training run declares a non-holdout CV, so it is not "
        "reportable as a holdout evaluation under the identity it carries"
    )
    _delete_holdout_generation(CASE_DIR, row["prediction_hash"])
superseded = list(retire.superseded)
if superseded and not REPLACE_HOLDOUT:
    raise RuntimeError(
        "the holdout window already carries a refit of a different configuration: "
        + ", ".join(
            f"{row['prediction_hash']} ({row['config_name']}, training {row['training_hash']})"
            for row in superseded
        )
        + f". This run would evaluate {carrier['config_name']} (training "
        f"{holdout_training_hash}, checkpoint {CHECKPOINT_KIND}={CHECKPOINT_VALUE}) on the "
        "same window. Set REPLACE_HOLDOUT=True to discard the earlier generation, or leave "
        "the selection where it was."
    )
for row in superseded:
    print(f"REPLACING holdout generation {row['prediction_hash']} ({row['config_name']})")
    _delete_holdout_generation(CASE_DIR, row["prediction_hash"])

In [8]:
request = reconstruct_locked_model_request(
    study,
    holdout_spec,
    checkpoint_kind=CHECKPOINT_KIND,
    checkpoint_value=CHECKPOINT_VALUE,
)
model_run = request.run()
holdout_prediction = model_run.predictions[0]

if model_run.training.hash == carrier["training_hash"]:
    raise RuntimeError(
        "the holdout refit produced the validation training identity "
        f"{carrier['training_hash']}, which means it did not refit"
    )
print(f"Holdout training run:  {model_run.training.hash}")
print(f"Holdout prediction set: {holdout_prediction.hash}")

Holdout training run:  37b83b09fa56
Holdout prediction set: dc47b209c6e4


What the prediction set covers, read back from the registry rather than from the
request. The two agree only if the fit published what it declared, and the count is the
one number a reader can check the window against: a monthly panel over one year is
twelve decision dates, and the number of rows is those dates times the names eligible on
each.

<!-- ML4T_ZH_BEGIN c0013-p000 -->
此處從登錄資料庫，而非請求內容，讀回預測集涵蓋的範圍。只有擬合實際發布了宣告的內容，兩者才會一致。計數是讀者能用來檢查視窗的數字：為期一年的月度面板有十二個決策日期，資料列數則是每個日期符合資格的股票檔數加總。
<!-- ML4T_ZH_END c0013-p000 -->


In [9]:
record = holdout_prediction.registry_record()
predictions = holdout_prediction.load()
print(
    f"split={record['split']}  checkpoint={record['checkpoint_kind']}={record['checkpoint_value']}"
)
print(f"rows={predictions.height:,}  dates={predictions['timestamp'].n_unique()}")
print(
    f"  {predictions['timestamp'].min()} -> {predictions['timestamp'].max()}, "
    f"{predictions['symbol'].n_unique():,} names"
)

split=holdout  checkpoint=iteration=500
rows=23,648  dates=12
  2016-01-29 -> 2016-12-30, 2,098 names


The registry now holds more than one holdout prediction set for this case study, and
only one of them was fitted on data that ends before the window. The other is the
defective generation this notebook replaces: it carries the validation training identity,
which is how it was found. Both are listed rather than one silently preferred, because
the registry is immutable and a reader looking at it later will see both.

<!-- ML4T_ZH_BEGIN c0015-p000 -->
登錄資料庫目前持有本案例研究不只一組保留測試集預測，但其中只有一組是使用在視窗開啟前就結束的資料擬合。另一組是本 Notebook 要取代的有缺陷版本：它帶有驗證期訓練識別，也因此得以辨認。這裡列出兩者，而非悄悄偏好其中之一，因為登錄資料庫不可變，日後查看的讀者會看到兩者。
<!-- ML4T_ZH_END c0015-p000 -->


In [10]:
for row in registered_holdout_generations(CASE_DIR):
    note = (
        "refitted for the holdout" if row["refitted"] else "VALIDATION-FITTED - not out of sample"
    )
    print(
        f"  {row['prediction_hash']}  training={row['training_hash']}  {row['config_name']}  {note}"
    )

  dc47b209c6e4  training=37b83b09fa56  leaves_31_huber  refitted for the holdout


## What this notebook establishes, and what it does not

It establishes one thing: a prediction set over the holdout window, produced by the
configuration this case study selected, fitted on data that ends before the window
opens. That is a precondition for an out-of-sample claim, not the claim itself. Nothing
here says whether the predictions are any good - they have not been scored, sized or
traded.

It does not make the holdout a fresh test in the strict sense. The configuration reached
this notebook through a selection made on the validation folds, and this window is being
used once per configuration that gets here. What it does remove is the specific
circularity of scoring a validation-fitted model on the period meant to judge it.

The holdout is re-runnable. If a later pass finds the selection was wrong, the answer is
to delete this generation and produce another, not to treat the first as spent.

**Next:** [`16_holdout_backtest`](16_holdout_backtest.ipynb).

<!-- ML4T_ZH_BEGIN c0017-p000 -->
## 本 Notebook 確立了什麼，以及尚未確立什麼

它確立一件事：本案例研究選定的組態，使用在保留測試視窗開啟前就結束的資料擬合，並產生涵蓋該視窗的預測集。這是提出樣本外主張的先決條件，而非主張本身。此處沒有說明預測好壞；它們尚未評分、設定部位大小或進行交易。

嚴格來說，它並未使保留測試集成為全新的測試。組態經由在驗證折上做出的篩選來到此處，而每個來到此處的組態都會使用這個視窗一次。它確實消除了一項特定循環問題：拿依驗證資料擬合的模型，在原本用來評判它的期間評分。

保留測試可以重新執行。若後續檢查發現先前選擇有誤，應刪除這一代結果並產生新結果，而非認定第一次使用後視窗就已耗盡。

**下一步：**[`16_holdout_backtest`](16_holdout_backtest.ipynb)。
<!-- ML4T_ZH_END c0017-p000 -->
